In [ ]:
import os
import sys
import pandas as pd

ruta_raiz = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ruta_raiz not in sys.path:
    sys.path.insert(0, ruta_raiz)

from src.ETL.common.minio_client import download_df_parquet

access_key = os.getenv("MINIO_ACCESS_KEY")
if access_key is None:
    raise AssertionError("MINIO_ACCESS_KEY no definida.")

secret_key = os.getenv("MINIO_SECRET_KEY")
if secret_key is None:
    raise AssertionError("MINIO_SECRET_KEY no definida.")

#cambio fechas
fecha_inicio = "2025-02-10"
fecha_fin    = "2025-02-15"
fechas = pd.date_range(fecha_inicio, fecha_fin).strftime("%Y-%m-%d").tolist()

dfs_s, dfs_e = [], []

for fecha in fechas:
    try:
        s = download_df_parquet(
            access_key=access_key,
            secret_key=secret_key,
            object_name=f"grupo5/cleaned/gtfs_clean_scheduled/date={fecha}/gtfs_scheduled_{fecha}.parquet",
        )
        s["service_date"] = fecha
        dfs_s.append(s)
    except Exception as e:
        print(f"  [scheduled] Sin datos para {fecha}: {e}")

    try:
        e = download_df_parquet(
            access_key=access_key,
            secret_key=secret_key,
            object_name=f"grupo5/cleaned/eventos_nyc/date={fecha}/eventos_{fecha}.parquet",
        )
        dfs_e.append(e)
    except Exception as e:
        print(f"  [eventos]   Sin datos para {fecha}: {e}")

df_s = pd.concat(dfs_s, ignore_index=True)
df_e = pd.concat(dfs_e, ignore_index=True)

print(f"¡Descarga exitosa! {len(fechas)} días")
print(f"  df_s: {len(df_s):,} filas | df_e: {len(df_e):,} filas")

  [eventos]   Sin datos para 2025-02-11: S3 operation failed; code: NoSuchKey, message: The specified key does not exist., resource: /pd1/grupo5/cleaned/eventos_nyc/date=2025-02-11/eventos_2025-02-11.parquet, request_id: 189B2647989001C0, host_id: b9279e7bc293d1e0f68ec1116dc0cc8a781f7f54250dbb3df2509efd311a9297, bucket_name: pd1, object_name: grupo5/cleaned/eventos_nyc/date=2025-02-11/eventos_2025-02-11.parquet
  [eventos]   Sin datos para 2025-02-13: S3 operation failed; code: NoSuchKey, message: The specified key does not exist., resource: /pd1/grupo5/cleaned/eventos_nyc/date=2025-02-13/eventos_2025-02-13.parquet, request_id: 189B2647F1BB6CB6, host_id: b9279e7bc293d1e0f68ec1116dc0cc8a781f7f54250dbb3df2509efd311a9297, bucket_name: pd1, object_name: grupo5/cleaned/eventos_nyc/date=2025-02-13/eventos_2025-02-13.parquet
¡Descarga exitosa! 6 días
  df_s: 973,592 filas | df_e: 68 filas


In [ ]:

df_s['hora_real_hhmm'] = pd.to_datetime(df_s['actual_seconds'], unit='s').dt.strftime('%H:%M:%S')
df_s['hora_programada_hhmm'] = pd.to_datetime(df_s['scheduled_seconds'], unit='s').dt.strftime('%H:%M:%S')
df_s['hora_referencia'] = df_s['hora_real_hhmm'].fillna(df_s['hora_programada_hhmm'])
df_s['tiempo_tren'] = pd.to_datetime(df_s['service_date'] + ' ' + df_s['hora_referencia'], errors='coerce')
df_e['inicio_dt'] = pd.to_datetime(df_e['fecha_inicio'] + ' ' + df_e['hora_inicio'], errors='coerce')
df_e['fin_dt'] = pd.to_datetime(df_e['fecha_final'] + ' ' + df_e['hora_salida_estimada'], errors='coerce')


df_s['trip_row_id'] = df_s.index


df_temp = pd.merge(
    df_s[['trip_row_id', 'stop_id', 'service_date', 'tiempo_tren']], 
    df_e[['stop_id', 'fecha_inicio', 'nombre_evento', 'tipo', 'score', 'inicio_dt', 'fin_dt']], 
    left_on=['stop_id', 'service_date'], 
    right_on=['stop_id', 'fecha_inicio'], 
    how='left'
)


ventana_entrada_inicio = df_temp['inicio_dt'] - pd.Timedelta(hours=1.5)
ventana_salida_fin = df_temp['fin_dt'] + pd.Timedelta(hours=1.5)


df_temp['durante_entrada'] = ((df_temp['tiempo_tren'] >= ventana_entrada_inicio) & (df_temp['tiempo_tren'] < df_temp['inicio_dt'])).astype(int)
df_temp['durante_evento'] = ((df_temp['tiempo_tren'] >= df_temp['inicio_dt']) & (df_temp['tiempo_tren'] <= df_temp['fin_dt'])).astype(int)
df_temp['despues_evento'] = ((df_temp['tiempo_tren'] > df_temp['fin_dt']) & (df_temp['tiempo_tren'] <= ventana_salida_fin)).astype(int)
df_temp['evento_nocturno'] = (df_temp['fin_dt'].dt.hour >= 22).astype(int)


df_temp = df_temp.sort_values(by=['trip_row_id', 'score'], ascending=[True, False])

df_agrupado = df_temp.groupby('trip_row_id').agg(
    hubo_evento_en_el_dia=('nombre_evento', lambda x: 1 if x.notna().any() else 0),
    n_eventos=('nombre_evento', 'nunique'),
    tipo_evento_prioritario=('tipo', 'first'), 
    durante_entrada=('durante_entrada', 'max'),
    durante_evento=('durante_evento', 'max'),
    despues_evento=('despues_evento', 'max'),
    evento_nocturno=('evento_nocturno', 'max')
).reset_index()


df_agrupado['tipo_evento_prioritario'] = df_agrupado['tipo_evento_prioritario'].fillna('Ninguno')


df_s = pd.merge(df_s, df_agrupado, on='trip_row_id', how='left')


df_s = df_s.drop(columns=['trip_row_id', 'hora_referencia', 'tiempo_tren', 'hora_real_hhmm', 'hora_programada_hhmm'])